# Multi-LLM-as-Judge: ADRD Medical AI
## Complete SMU SuperPOD Experiment Notebook

**Tailored to:** `mkotha` on SMU SuperPOD | Account: `xnluo_ai_chatbot_cognitive_0002`

| Phase | Cell | What happens |
|---|---|---|
| 0 | — | SSH + SOCKS proxy + tmux + srun (local terminal) |
| 1 | 1 | Verify GPU node, venv, paths |
| 2 | 2 | Install deps + tabulate fix |
| 3 | 3 | Load .env + HF cache |
| 3 | 4 | Download all 4 judge models |
| 4 | 5 | Kill any old vLLM + launch 4 fresh servers |
| 4 | 6 | Tail logs (optional check) |
| 5 | 7 | Health check all 4 judges (20 min timeout) |
| 6 | 8 | Repo structure check |
| 6 | 9 | Unit tests |
| 7 | 10 | Exp 1 — Dataset analysis |
| 8 | 11 | Exp 2 — Agreement analysis |
| 9 | 12 | Exp 3 — Rubric sensitivity |
| 10 | 13 | Exp 4 — Box plots |
| 11 | 14 | Results summary + CSV |
| 12 | 15 | Display figures inline |
| 13 | 16 | Push to GitHub |
| 14 | 17 | Cleanup / kill vLLM |

---

## Phase 0 — Terminal Commands (run BEFORE opening this notebook)

```bash
# 1. On your LOCAL machine:
ssh -C -D 8080 mkotha@superpod.smu.edu

# 2. On SuperPOD login node:
tmux new -s cs7325
# (to reattach later: tmux attach -t cs7325)

# 3. Request GPU node (inside tmux):
srun -A xnluo_ai_chatbot_cognitive_0002 -N1 -G2 -c32 --mem=192G --time=4:00:00 --pty $SHELL

# 4. On the GPU compute node:
cd /lustre/smuexa01/client/users/mkotha/CS7325

# 5. Clone repo (only first time):
git clone https://YOUR_GITHUB_TOKEN@github.com/m22oct2000/Multi_LLM_as_Judge_Medical_AI.git
cd Multi_LLM_as_Judge_Medical_AI

# 6. Create .env file (only first time):
cat > .env << 'EOF'
HF_TOKEN=hf_xxxxxxxxxxxx
GITHUB_TOKEN=ghp_xxxxxxxxxxxx
LOCAL_VLLM_JUDGE1_URL=http://localhost:8001
LOCAL_VLLM_JUDGE2_URL=http://localhost:8002
LOCAL_VLLM_JUDGE3_URL=http://localhost:8003
LOCAL_VLLM_JUDGE4_URL=http://localhost:8004
EOF

# 7. Activate venv and launch JupyterLab:
source /lustre/smuexa01/client/users/mkotha/CS7325/.venv/bin/activate
cd /lustre/smuexa01/client/users/mkotha/CS7325/Multi_LLM_as_Judge_Medical_AI
jupyter lab --ip=0.0.0.0 --no-browser
# Open the printed URL in your browser (SOCKS proxy port 8080)
```


---
## Phase 1 — Verify GPU Node & Environment

In [1]:
# CELL 1: Verify GPU, venv, paths
import subprocess, os, sys, importlib
from pathlib import Path
from datetime import datetime

def shell(cmd, check=False):
    result = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    out = (result.stdout + result.stderr).strip()
    if out: print(out)
    if check and result.returncode != 0:
        raise RuntimeError(f'Command failed: {cmd}')
    return result.returncode

LUSTRE_BASE = Path('/lustre/smuexa01/client/users/mkotha/CS7325')
ROOT        = LUSTRE_BASE / 'Multi_LLM_as_Judge_Medical_AI'
VENV        = LUSTRE_BASE / '.venv'
HF_CACHE    = LUSTRE_BASE / 'hf_models'
LOG_DIR     = LUSTRE_BASE / 'vllm_logs'
PYTHON      = str(VENV / 'bin' / 'python')

HF_CACHE.mkdir(parents=True, exist_ok=True)
LOG_DIR.mkdir(parents=True, exist_ok=True)

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

print(f'Repo root : {ROOT}')
print(f'venv      : {VENV}')
print(f'HF cache  : {HF_CACHE}')
print(f'Python    : {sys.executable}')
print(f'Started   : {datetime.now().isoformat()}')

print('\n=== GPU Status (expect 2x A100 80GB) ===')
shell('nvidia-smi --query-gpu=index,name,memory.total,memory.free --format=csv,noheader')

print('\n=== SLURM allocation ===')
shell('echo "Node: $SLURMD_NODENAME  Job: $SLURM_JOB_ID  GPUs: $SLURM_GPUS  CPUs: $SLURM_CPUS_ON_NODE  Mem: $SLURM_MEM_PER_NODE"')

print('\n=== venv active? ===')
venv_ok = str(VENV) in sys.executable
print(f'  {"✅ YES" if venv_ok else "❌ NO — run: source " + str(VENV) + "/bin/activate"}')

print('\n=== Lustre disk ===')
shell(f'df -h {LUSTRE_BASE} | tail -1')

Repo root : /lustre/smuexa01/client/users/mkotha/CS7325/Multi_LLM_as_Judge_Medical_AI
venv      : /lustre/smuexa01/client/users/mkotha/CS7325/.venv
HF cache  : /lustre/smuexa01/client/users/mkotha/CS7325/hf_models
Python    : /lustre/smuexa01/client/users/mkotha/CS7325/.venv/bin/python3
Started   : 2026-05-17T22:05:35.171769

=== GPU Status (expect 2x A100 80GB) ===
0, NVIDIA A100-SXM4-80GB, 81920 MiB, 81154 MiB
1, NVIDIA A100-SXM4-80GB, 81920 MiB, 81154 MiB
2, NVIDIA A100-SXM4-80GB, 81920 MiB, 81154 MiB
3, NVIDIA A100-SXM4-80GB, 81920 MiB, 81154 MiB

=== SLURM allocation ===
Node: bcm-dgxa100-0014  Job: 433186  GPUs: 4  CPUs: 64  Mem: 196608

=== venv active? ===
  ✅ YES

=== Lustre disk ===
100.127.24.16@o2ib1,100.127.25.16@o2ib1:100.127.24.17@o2ib1,100.127.25.17@o2ib1:100.127.24.18@o2ib1,100.127.25.18@o2ib1:100.127.24.19@o2ib1,100.127.25.19@o2ib1:/smuexa01  748T   37T  703T   5% /lustre/smuexa01/client


0

---
## Phase 2 — Install Dependencies

In [2]:
# CELL 2: Install all deps + auto-fix missing packages
REQUIREMENTS = ROOT / 'requirements.txt'
print(f'Installing from: {REQUIREMENTS}')
shell(f'{sys.executable} -m pip install -r {REQUIREMENTS} -q')

print('\nInstalling / verifying vLLM...')
shell(f'{sys.executable} -m pip install vllm -q')

# Auto-fix known missing packages
ALWAYS_INSTALL = ['tabulate', 'ipywidgets']
for pkg in ALWAYS_INSTALL:
    shell(f'{sys.executable} -m pip install {pkg} -q')

# Verify
REQUIRED = ['vllm', 'httpx', 'datasets', 'huggingface_hub', 'pandas',
            'matplotlib', 'seaborn', 'dotenv', 'pytest', 'tabulate']
print('\n=== Package check ===')
all_ok = True
for pkg in REQUIRED:
    try:
        m = importlib.import_module(pkg.replace('-', '_'))
        ver = getattr(m, '__version__', '?')
        print(f'  ✅ {pkg:<20} {ver}')
    except ImportError:
        print(f'  ❌ {pkg} — FAILED')
        all_ok = False

print('\n✅ All packages OK' if all_ok else '\n❌ Some packages missing — re-run this cell')

Installing from: /lustre/smuexa01/client/users/mkotha/CS7325/Multi_LLM_as_Judge_Medical_AI/requirements.txt

Installing / verifying vLLM...

=== Package check ===
INFO 05-17 22:05:48 [__init__.py:239] Automatically detected platform cuda.
  ✅ vllm                 0.8.5
  ✅ httpx                0.28.1
  ✅ datasets             4.8.5
  ✅ huggingface_hub      0.30.2
  ✅ pandas               2.3.3
  ✅ matplotlib           3.10.9
  ✅ seaborn              0.13.2
  ✅ dotenv               ?
  ✅ pytest               9.0.3
  ✅ tabulate             0.10.0

✅ All packages OK


--- 
## Phase 3 — Download Judge Models

Models cached at `/lustre/.../hf_models/` — never fills `/home`.

| Judge | Model | Max context | GPU | Notes |
|---|---|---|---|---|
| medgemma  | `google/medgemma-4b-it`    | 4096 | GPU 0 | Gated — HF token required |
| biomistral| `BioMistral/BioMistral-7B` | 4096 | GPU 1 | |
| meditron  | `epfl-llm/meditron-7b`     | 2048 | GPU 0 | |
| medalpaca | `medalpaca/medalpaca-7b`   | 2048 | GPU 1 | Replaces BioMedLM (vLLM 0.8.5 incompatible) |


In [3]:
# CELL 3: Load .env + set HF cache
from dotenv import load_dotenv
load_dotenv(ROOT / '.env')

os.environ['HF_HOME']            = str(HF_CACHE)
os.environ['TRANSFORMERS_CACHE'] = str(HF_CACHE)
os.environ['HF_DATASETS_CACHE']  = str(HF_CACHE / 'datasets')

HF_TOKEN = os.environ.get('HF_TOKEN')
GH_TOKEN = os.environ.get('GITHUB_TOKEN')

print(f'HF_TOKEN:      {"✅ set" if HF_TOKEN else "❌ NOT SET — add to .env"}')
print(f'GITHUB_TOKEN:  {"✅ set" if GH_TOKEN else "⚠️  not set — needed for git push"}')
print(f'HF_CACHE:      {HF_CACHE}')
shell(f'du -sh {HF_CACHE} 2>/dev/null || echo "(empty)"')

HF_TOKEN:      ✅ set
GITHUB_TOKEN:  ✅ set
HF_CACHE:      /lustre/smuexa01/client/users/mkotha/CS7325/hf_models
179G	/lustre/smuexa01/client/users/mkotha/CS7325/hf_models


0

In [4]:
# CELL 4: Download all judge models (skips already-cached)
# NOTE: BioMedLM (stanford-crfm/BioMedLM) is NOT downloaded —
#       it has a hard incompatibility with vLLM 0.8.5
#       (AssertionError: scale_attn_by_inverse_layer_idx not supported).
#       medalpaca/medalpaca-7b is used instead as Judge 4.
from huggingface_hub import snapshot_download
import time

JUDGE_MODELS = [
    {'id': 'medgemma',   'hf_id': 'google/medgemma-4b-it',       'needs_token': True},
    {'id': 'biomistral', 'hf_id': 'BioMistral/BioMistral-7B',    'needs_token': False},
    {'id': 'meditron',   'hf_id': 'epfl-llm/meditron-7b',        'needs_token': False},
    {'id': 'medalpaca',  'hf_id': 'medalpaca/medalpaca-7b',       'needs_token': False},
    # BioMedLM removed — incompatible with vLLM 0.8.5
    # {'id': 'biomedlm', 'hf_id': 'stanford-crfm/BioMedLM', ...}
]

for m in JUDGE_MODELS:
    marker_dir = HF_CACHE / f'models--{m["hf_id"].replace("/", "--")}'
    if marker_dir.exists() and any(marker_dir.rglob('config.json')):
        print(f'✅ [{m["id"]:12s}] already cached → {marker_dir}')
        continue
    if m['needs_token'] and not HF_TOKEN:
        print(f'❌ [{m["id"]:12s}] requires HF_TOKEN — set in .env')
        continue
    print(f'\n⏬  Downloading [{m["id"]}] {m["hf_id"]} ...')
    t0 = time.time()
    try:
        snapshot_download(
            repo_id=m['hf_id'],
            cache_dir=str(HF_CACHE),
            token=HF_TOKEN if m['needs_token'] else None,
            ignore_patterns=['*.msgpack', '*.h5', 'flax_model*', 'tf_model*', 'rust_model*'],
        )
        print(f'  ✅ Done in {(time.time()-t0)/60:.1f} min')
    except Exception as e:
        print(f'  ❌ Failed: {e}')

print('\n=== Total model cache ===')
shell(f'du -sh {HF_CACHE}')


✅ [medgemma    ] already cached → /lustre/smuexa01/client/users/mkotha/CS7325/hf_models/models--google--medgemma-4b-it
✅ [biomistral  ] already cached → /lustre/smuexa01/client/users/mkotha/CS7325/hf_models/models--BioMistral--BioMistral-7B
✅ [meditron    ] already cached → /lustre/smuexa01/client/users/mkotha/CS7325/hf_models/models--epfl-llm--meditron-7b
✅ [medalpaca   ] already cached → /lustre/smuexa01/client/users/mkotha/CS7325/hf_models/models--medalpaca--medalpaca-7b

=== Total model cache ===
179G	/lustre/smuexa01/client/users/mkotha/CS7325/hf_models


0

---
## Phase 4 — Launch vLLM Servers

**4x A100 80GB — 1 dedicated GPU per model, all launched in parallel.**

- `--enforce-eager` skips CUDA graph compilation → ~45s startup per model (was ~5min)
- All 4 start simultaneously → total ready time ~90s (was ~20min)
- `HF_HUB_OFFLINE=1` + local snapshot paths → no HF auth needed at runtime
- BioMedLM → medalpaca (BioMedLM incompatible with vLLM 0.8.5)

| Judge | Model | GPU | Port | Max Len |
|---|---|---|---|---|
| medgemma  | `google/medgemma-4b-it`    | GPU 0 | 8001 | 4096 |
| biomistral| `BioMistral/BioMistral-7B` | GPU 1 | 8002 | 4096 |
| meditron  | `epfl-llm/meditron-7b`     | GPU 2 | 8003 | 2048 |
| medalpaca | `medalpaca/medalpaca-7b`   | GPU 3 | 8004 | 2048 |


In [5]:
# CELL 5: Launch all 4 vLLM judges in parallel (fast startup)
import time, subprocess, os, urllib.request
from pathlib import Path

LUSTRE_BASE = Path('/lustre/smuexa01/client/users/mkotha/CS7325')
HF_CACHE    = LUSTRE_BASE / 'hf_models'
LOG_DIR     = LUSTRE_BASE / 'vllm_logs'
PYTHON      = str(LUSTRE_BASE / '.venv' / 'bin' / 'python')
LOG_DIR.mkdir(parents=True, exist_ok=True)

# ── Step 1: Full cleanup ─────────────────────────────────────────────────────
print('=' * 60)
print('Step 1: Killing any existing vLLM processes...')
subprocess.run('pkill -9 -f "vllm.entrypoints" 2>/dev/null || true', shell=True)
time.sleep(3)
subprocess.run(
    'kill -9 $(fuser /dev/nvidia0 /dev/nvidia1 /dev/nvidia2 /dev/nvidia3 2>/dev/null) 2>/dev/null || true',
    shell=True
)
time.sleep(5)
print('GPU memory after cleanup:')
subprocess.run('nvidia-smi --query-gpu=index,memory.used,memory.free --format=csv,noheader', shell=True)

# ── Step 2: Resolve local snapshot paths ────────────────────────────────────
print('\nStep 2: Resolving local snapshot paths...')

def get_snapshot_path(hf_id):
    cache_name = 'models--' + hf_id.replace('/', '--')
    snap_dir = HF_CACHE / cache_name / 'snapshots'
    if not snap_dir.exists():
        raise FileNotFoundError(f"Not cached: {hf_id} — run Cell 4 first.")
    snapshots = list(snap_dir.iterdir())
    if not snapshots:
        raise FileNotFoundError(f"Empty snapshot dir for {hf_id}")
    return str(sorted(snapshots, key=lambda p: p.stat().st_mtime)[-1])

# 4x A100 80GB — 1 dedicated GPU per model
# --enforce-eager skips CUDA graph compilation: startup ~45s vs ~5min
JUDGE_CONFIGS = [
    {'id': 'medgemma',   'hf_id': 'google/medgemma-4b-it',    'port': 8001, 'gpu': '0', 'max_len': 4096, 'extra': '--trust-remote-code --enforce-eager'},
    {'id': 'biomistral', 'hf_id': 'BioMistral/BioMistral-7B', 'port': 8002, 'gpu': '1', 'max_len': 4096, 'extra': '--enforce-eager'},
    {'id': 'meditron',   'hf_id': 'epfl-llm/meditron-7b',     'port': 8003, 'gpu': '2', 'max_len': 2048, 'extra': '--enforce-eager'},
    {'id': 'medalpaca',  'hf_id': 'medalpaca/medalpaca-7b',    'port': 8004, 'gpu': '3', 'max_len': 2048, 'extra': '--enforce-eager'},
]

snapshot_paths = {}
all_cached = True
for j in JUDGE_CONFIGS:
    try:
        snap = get_snapshot_path(j['hf_id'])
        snapshot_paths[j['id']] = snap
        print(f"  ✅ [{j['id']:12s}] found")
    except FileNotFoundError as e:
        print(f"  ❌ [{j['id']:12s}] {e}")
        all_cached = False

if not all_cached:
    raise RuntimeError("Some models not cached — run Cell 4 first.")

# ── Step 3: Launch ALL 4 in parallel ────────────────────────────────────────
print('\nStep 3: Launching all 4 judges in PARALLEL...')
print('--enforce-eager: skips CUDA graph compile → ready in ~60-90s total\n')

procs = {}
for j in JUDGE_CONFIGS:
    log_file = str(LOG_DIR / f'vllm_{j["id"]}.log')
    subprocess.run(f'fuser -k {j["port"]}/tcp 2>/dev/null || true', shell=True)

    cmd = (
        f'CUDA_VISIBLE_DEVICES={j["gpu"]} '
        f'HF_HUB_OFFLINE=1 TRANSFORMERS_OFFLINE=1 '
        f'HF_HOME={HF_CACHE} '
        f'{PYTHON} -m vllm.entrypoints.openai.api_server '
        f'--model {snapshot_paths[j["id"]]} '
        f'--port {j["port"]} --host 0.0.0.0 '
        f'--gpu-memory-utilization 0.85 '
        f'--max-model-len {j["max_len"]} '
        f'--dtype bfloat16 '
        f'{j["extra"]} '
        f'> {log_file} 2>&1'
    )
    proc = subprocess.Popen(cmd, shell=True)
    procs[j['id']] = proc
    print(f'  🚀 [{j["id"]:12s}] PID={proc.pid}  GPU={j["gpu"]}  port={j["port"]}')

JUDGE_URLS = {j['id']: f'http://localhost:{j["port"]}' for j in JUDGE_CONFIGS}
print('\nAll 4 launched simultaneously. Run Cell 7 to wait for health checks (~90s).')


Step 1: Killing any existing vLLM processes...
GPU memory after cleanup:
0, 0 MiB, 81154 MiB
1, 0 MiB, 81154 MiB
2, 0 MiB, 81154 MiB
3, 0 MiB, 81154 MiB

Step 2: Resolving local snapshot paths...
  ✅ [medgemma    ] found
  ✅ [biomistral  ] found
  ✅ [meditron    ] found
  ✅ [medalpaca   ] found

Step 3: Launching all 4 judges in PARALLEL...
--enforce-eager: skips CUDA graph compile → ready in ~60-90s total

  🚀 [medgemma    ] PID=2350624  GPU=0  port=8001
  🚀 [biomistral  ] PID=2350628  GPU=1  port=8002
  🚀 [meditron    ] PID=2350632  GPU=2  port=8003
  🚀 [medalpaca   ] PID=2350637  GPU=3  port=8004

All 4 launched simultaneously. Run Cell 7 to wait for health checks (~90s).


In [6]:
# CELL 6 (optional): Check startup progress — run anytime while waiting
import subprocess
from pathlib import Path

LOG_DIR = Path('/lustre/smuexa01/client/users/mkotha/CS7325/vllm_logs')
JUDGES  = [
    {'id': 'medgemma',   'gpu': '0', 'port': 8001},
    {'id': 'biomistral', 'gpu': '1', 'port': 8002},
    {'id': 'meditron',   'gpu': '2', 'port': 8003},
    {'id': 'medalpaca',  'gpu': '3', 'port': 8004},
]

for j in JUDGES:
    log_file = LOG_DIR / f'vllm_{j["id"]}.log'
    print(f'\n--- [{j["id"]}] GPU={j["gpu"]} port={j["port"]} ---')
    subprocess.run(f'tail -4 {log_file} 2>/dev/null || echo "(no log yet)"', shell=True)

print('\n=== GPU memory ===')
subprocess.run('nvidia-smi --query-gpu=index,memory.used,memory.free --format=csv,noheader', shell=True)



--- [medgemma] GPU=0 port=8001 ---

--- [biomistral] GPU=1 port=8002 ---

--- [meditron] GPU=2 port=8003 ---

--- [medalpaca] GPU=3 port=8004 ---

=== GPU memory ===
0, 0 MiB, 81154 MiB
1, 0 MiB, 81154 MiB
2, 0 MiB, 81154 MiB
3, 0 MiB, 81154 MiB


CompletedProcess(args='nvidia-smi --query-gpu=index,memory.used,memory.free --format=csv,noheader', returncode=0)

---
## Phase 5 — Health Check (wait for all 4 judges)

In [7]:
# CELL 7: Wait for all 4 judges — polls every 10s, 10 min max
import httpx, time, subprocess
from pathlib import Path

LOG_DIR = Path('/lustre/smuexa01/client/users/mkotha/CS7325/vllm_logs')

if 'JUDGE_URLS' not in dir():
    JUDGE_URLS = {
        'medgemma':   'http://localhost:8001',
        'biomistral': 'http://localhost:8002',
        'meditron':   'http://localhost:8003',
        'medalpaca':  'http://localhost:8004',
    }

MAX_WAIT = 600
POLL     = 10
ready    = {k: False for k in JUDGE_URLS}
ready_time = {}
t_start  = time.time()

print('Waiting for all 4 vLLM judges (--enforce-eager = ~60-90s startup)...\n')

while not all(ready.values()):
    elapsed = time.time() - t_start
    if elapsed > MAX_WAIT:
        print(f'\n⏱️  Timeout after {elapsed/60:.1f} min. Error summary:')
        for name in [k for k, v in ready.items() if not v]:
            log = LOG_DIR / f'vllm_{name}.log'
            print(f'\n--- [{name}] errors ---')
            subprocess.run(
                f'grep -i "error\|Error\|assert\|OOM\|No available" {log} | tail -8',
                shell=True
            )
        break

    for name, url in JUDGE_URLS.items():
        if ready[name]:
            continue
        try:
            r = httpx.get(f'{url}/health', timeout=3)
            if r.status_code == 200:
                ready[name] = True
                ready_time[name] = time.time() - t_start
                print(f'  ✅ {name:12s} ready in {ready_time[name]:.0f}s')
        except Exception:
            pass

    if not all(ready.values()):
        pending = [k for k, v in ready.items() if not v]
        print(f'  ⏳ [{time.time()-t_start:5.0f}s] Pending: {pending}', end='\r')
        time.sleep(POLL)

print()
if all(ready.values()):
    total = time.time() - t_start
    print(f'\n✅ ALL 4 JUDGES READY in {total:.0f}s ({total/60:.1f} min)')
    print('\nGPU memory:')
    subprocess.run('nvidia-smi --query-gpu=index,memory.used,memory.free --format=csv,noheader', shell=True)
    print('\n→ Proceed to Cell 8')
else:
    failed = [k for k, v in ready.items() if not v]
    print(f'\n❌ Failed: {failed} — check logs above, re-run Cell 5')


Waiting for all 4 vLLM judges (--enforce-eager = ~60-90s startup)...

  ✅ biomistral   ready in 51smma', 'biomistral', 'meditron', 'medalpaca']
  ✅ meditron     ready in 51s
  ✅ medalpaca    ready in 51s
  ✅ medgemma     ready in 61smma']


✅ ALL 4 JUDGES READY in 61s (1.0 min)

GPU memory:
0, 68201 MiB, 12953 MiB
1, 69119 MiB, 12035 MiB
2, 69101 MiB, 12053 MiB
3, 69101 MiB, 12053 MiB

→ Proceed to Cell 8


---
## Phase 6 — Pre-Flight Checks

In [8]:
# CELL 8: Verify all required repo files exist
import json

REQUIRED_PATHS = [
    ROOT / 'core'        / 'wrapper.py',
    ROOT / 'core'        / 'agreement.py',
    ROOT / 'core'        / 'rubric_engine.py',
    ROOT / 'core'        / 'metrics.py',
    ROOT / 'core'        / 'schemas.py',
    ROOT / 'core'        / 'consensus_core' / 'models.py',
    ROOT / 'rubrics'     / 'rubric1_pemat.json',
    ROOT / 'rubrics'     / 'rubric2_healthbench.json',
    ROOT / 'rubrics'     / 'rubric3_clinical_eval.json',
    ROOT / 'rubrics'     / 'rubric4_prometheus.json',
    ROOT / 'config'      / 'configs' / 'config_exp1_dataset.json',
    ROOT / 'config'      / 'configs' / 'config_exp2_agreement.json',
    ROOT / 'experiments' / 'exp1_dataset_analysis.py',
    ROOT / 'experiments' / 'exp2_agreement_analysis.py',
    ROOT / 'experiments' / 'exp3_rubric_sensitivity.py',
    ROOT / 'experiments' / 'exp4_boxplot_agreement.py',
]

print('=== Repo structure check ===')
all_ok = True
for p in REQUIRED_PATHS:
    ok = p.exists()
    print(f'  {"✅" if ok else "❌"} {p.relative_to(ROOT)}')
    if not ok: all_ok = False

if not all_ok:
    print(f'\n❌ Missing files — run: git -C {ROOT} pull origin main')
else:
    print('\n✅ All files present')

=== Repo structure check ===
  ✅ core/wrapper.py
  ✅ core/agreement.py
  ✅ core/rubric_engine.py
  ✅ core/metrics.py
  ✅ core/schemas.py
  ✅ core/consensus_core/models.py
  ✅ rubrics/rubric1_pemat.json
  ✅ rubrics/rubric2_healthbench.json
  ✅ rubrics/rubric3_clinical_eval.json
  ✅ rubrics/rubric4_prometheus.json
  ✅ config/configs/config_exp1_dataset.json
  ✅ config/configs/config_exp2_agreement.json
  ✅ experiments/exp1_dataset_analysis.py
  ✅ experiments/exp2_agreement_analysis.py
  ✅ experiments/exp3_rubric_sensitivity.py
  ✅ experiments/exp4_boxplot_agreement.py

✅ All files present


In [9]:
# CELL 9: Run unit tests
print('Running pytest...')
rc = shell(f'{sys.executable} -m pytest {ROOT}/tests/test_core.py -v --tb=short')
print('\n✅ All tests passed' if rc == 0 else '\n⚠️  Tests failed — review before running experiments')

Running pytest...
============================= test session starts ==============================
platform linux -- Python 3.10.12, pytest-9.0.3, pluggy-1.6.0 -- /lustre/smuexa01/client/users/mkotha/CS7325/.venv/bin/python3
cachedir: .pytest_cache
rootdir: /lustre/smuexa01/client/users/mkotha/CS7325
configfile: pyproject.toml
plugins: anyio-4.13.0
collecting ... collected 6 items

../tests/test_core.py::test_rubrics PASSED                               [ 16%]
../tests/test_core.py::test_configs PASSED                               [ 33%]
../tests/test_core.py::test_benchmark_csv PASSED                         [ 50%]
../tests/test_core.py::test_heuristics PASSED                            [ 66%]
../tests/test_core.py::test_exp1_placeholder PASSED                      [ 83%]
../tests/test_core.py::test_agreement_math PASSED                        [100%]

=============================== warnings summary ===============================
../../.venv/lib/python3.10/site-packages/_pytest/conf

---
## Phase 7 — Experiment 1: ADRD-Bench Dataset Analysis

In [10]:
# CELL 10: Experiment 1
import importlib, experiments.exp1_dataset_analysis as exp1_mod
importlib.reload(exp1_mod)

print('Running Experiment 1: ADRD-Bench Dataset Analysis')
print('='*60)
exp1_mod.main()

from collections import Counter
import pandas as pd
q_path = ROOT / 'benchmark_dataset' / 'adrd_questions.json'
with open(q_path) as f:
    qdata = json.load(f)
counts = Counter(q['category'] for q in qdata['questions'])
df_cat = pd.DataFrame([{'Category': k, 'N': v} for k, v in sorted(counts.items())])
df_cat.loc[len(df_cat)] = ['TOTAL', df_cat['N'].sum()]
print('\n=== Dataset Breakdown ===')
print(df_cat.to_markdown(index=False))
print(f'\n✅ Exp1 complete — {qdata["metadata"]["total_questions"]} questions')

Running Experiment 1: ADRD-Bench Dataset Analysis
Experiment 1: Structured Multi-Domain Medical Benchmark
Sources: MedQuAD | MedDialog | Medical Meadow Health Advice
Origin: https://github.com/m22oct2000/Multi-LLMs-as-Judge
Data dir: /lustre/smuexa01/client/users/mkotha/CS7325/Multi_LLM_as_Judge_Medical_AI/benchmark_dataset/source_datasets
Target per domain: 20


Total questions from real data: 0
Final benchmark size: 25 questions

Domain breakdown:
  Cardiology     :   5 questions
  Pharmacology   :   5 questions
  Neurology      :   5 questions
  Pediatrics     :   5 questions
  Emergency      :   5 questions

Source breakdown:
  placeholder         :  25

BENCHMARK TABLE (for paper):
Domain          Question Summary                                           #
----------------------------------------------------------------------------
Cardiology      STEMI management, 65-year-old, inferior ST elevation       5
Pharmacology    Metformin contraindications and renal safety criteria    

---
## Phase 8 — Experiment 2: Per-Rubric Agreement Analysis

In [11]:
# CELL 11: Experiment 2
import experiments.exp2_agreement_analysis as exp2_mod
importlib.reload(exp2_mod)

print('Running Experiment 2: Per-Rubric Agreement Analysis')
print('='*60)
exp2_mod.main()

exp2_path = ROOT / 'results' / 'exp2_agreement_results.json'
with open(exp2_path) as f:
    exp2_data = json.load(f)

rows = []
for block in exp2_data:
    results = block['results']
    classes = Counter(r['agreement_class'] for r in results)
    total   = len(results)
    mean_pw = sum(r['mean_pairwise_agreement'] for r in results) / total
    rows.append({
        'Rubric':           block['rubric_name'].split('—')[0].strip()[:30],
        'N':                total,
        'Fully Agree %':    round(classes.get('fully_agree',    0)/total*100, 1),
        'Majority Agree %': round(classes.get('majority_agree', 0)/total*100, 1),
        'Split %':          round(classes.get('split',          0)/total*100, 1),
        'Disagree %':       round(classes.get('full_disagree',  0)/total*100, 1),
        'Mean PW%':         round(mean_pw, 1),
    })
df_exp2 = pd.DataFrame(rows)
print('\n=== Agreement Table ===')
print(df_exp2.to_markdown(index=False))
print(f'\n✅ Exp2 complete')

2026-05-17 22:07:03,667 - adrd_judge_runner - INFO - ADRDJudgeRunner: 4 judges
2026-05-17 22:07:03,667 - adrd_judge_runner - INFO -   medgemma | google/medgemma-4b-it | http://localhost:8001
2026-05-17 22:07:03,668 - adrd_judge_runner - INFO -   biomistral | BioMistral/BioMistral-7B | http://localhost:8002
2026-05-17 22:07:03,668 - adrd_judge_runner - INFO -   meditron | epfl-llm/meditron-7b | http://localhost:8003
2026-05-17 22:07:03,668 - adrd_judge_runner - INFO -   biomedlm | stanford-crfm/BioMedLM | http://localhost:8004


Running Experiment 2: Per-Rubric Agreement Analysis
Loaded 15 questions from agreement benchmark CSV
Resuming: 5 rubric(s) already done: {'rubric1_pemat', 'rubric5_pemat_likert', 'rubric3_clinical_eval', 'rubric2_healthbench', 'rubric4_prometheus'}

Skipping already-done rubric: rubric1_pemat

Skipping already-done rubric: rubric2_healthbench

Skipping already-done rubric: rubric3_clinical_eval

Skipping already-done rubric: rubric4_prometheus

Skipping already-done rubric: rubric5_pemat_likert

Updated agreement_benchmark.csv with observed_class (verified rows: 0/15)

AGREEMENT SUMMARY

PEMAT — Patient Education Materials Assessment Tool:
  fully_agree         :   0/0 (0.0%)
  majority_agree      :   0/0 (0.0%)
  split               :   0/0 (0.0%)
  full_disagree       :   0/0 (0.0%)
  skipped             :  15 rows (excluded from %)
  Mean pairwise : 0.0%

HealthBench — Physician-Written Medical Response Evaluation Criteria:
  fully_agree         :   0/0 (0.0%)
  majority_agree      

---
## Phase 9 — Experiment 3: Rubric Sensitivity

In [12]:
# CELL 12: Experiment 3
import experiments.exp3_rubric_sensitivity as exp3_mod
importlib.reload(exp3_mod)

print('Running Experiment 3: Rubric Sensitivity')
print('='*60)
exp3_mod.main()

exp3_path = ROOT / 'results' / 'exp3_sensitivity_results.json'
with open(exp3_path) as f:
    exp3_data = json.load(f)

sens_rows = []
for block in exp3_data:
    for v in block['variants']:
        sens_rows.append({
            'Rubric':  block['rubric_name'].split('—')[0].strip()[:25],
            'Variant': v['scoring_variant'],
            'Mean%':   v['mean_pairwise_agreement'],
            'Std%':    v['std_pairwise_agreement'],
        })
print(pd.DataFrame(sens_rows).to_markdown(index=False))
print(f'\n✅ Exp3 complete')

2026-05-17 22:08:03,976 - adrd_judge_runner - INFO - ADRDJudgeRunner: 4 judges
2026-05-17 22:08:03,976 - adrd_judge_runner - INFO -   medgemma | google/medgemma-4b-it | http://localhost:8001
2026-05-17 22:08:03,976 - adrd_judge_runner - INFO -   biomistral | BioMistral/BioMistral-7B | http://localhost:8002
2026-05-17 22:08:03,977 - adrd_judge_runner - INFO -   meditron | epfl-llm/meditron-7b | http://localhost:8003
2026-05-17 22:08:03,977 - adrd_judge_runner - INFO -   biomedlm | stanford-crfm/BioMedLM | http://localhost:8004
2026-05-17 22:08:04,019 - httpx - INFO - HTTP Request: GET http://localhost:8001/v1/models "HTTP/1.1 200 OK"
2026-05-17 22:08:04,059 - httpx - INFO - HTTP Request: GET http://localhost:8002/v1/models "HTTP/1.1 200 OK"
2026-05-17 22:08:04,102 - httpx - INFO - HTTP Request: GET http://localhost:8003/v1/models "HTTP/1.1 200 OK"
2026-05-17 22:08:04,146 - httpx - INFO - HTTP Request: GET http://localhost:8004/v1/models "HTTP/1.1 200 OK"
2026-05-17 22:08:04,147 - adrd_j

Running Experiment 3: Rubric Sensitivity

Rubric: PEMAT — Patient Education Materials Assessment Tool | Variant: BINARY


2026-05-17 22:08:04,308 - httpx - INFO - HTTP Request: POST http://localhost:8003/v1/completions "HTTP/1.1 404 Not Found"
2026-05-17 22:08:04,310 - httpx - INFO - HTTP Request: POST http://localhost:8002/v1/completions "HTTP/1.1 404 Not Found"
2026-05-17 22:08:04,310 - adrd_judge_runner - ERROR - Judge meditron failed: Client error '404 Not Found' for url 'http://localhost:8003/v1/completions'
For more information check: https://developer.mozilla.org/en-US/docs/Web/HTTP/Status/404
2026-05-17 22:08:04,311 - httpx - INFO - HTTP Request: POST http://localhost:8001/v1/chat/completions "HTTP/1.1 404 Not Found"
2026-05-17 22:08:04,312 - adrd_judge_runner - ERROR - Judge biomistral failed: Client error '404 Not Found' for url 'http://localhost:8002/v1/completions'
For more information check: https://developer.mozilla.org/en-US/docs/Web/HTTP/Status/404
2026-05-17 22:08:04,313 - httpx - INFO - HTTP Request: POST http://localhost:8004/v1/completions "HTTP/1.1 404 Not Found"
2026-05-17 22:08:04,3

  [SKIPPED] Q=q_0000 — only 0 non-empty responses


2026-05-17 22:08:04,602 - httpx - INFO - HTTP Request: POST http://localhost:8002/v1/completions "HTTP/1.1 404 Not Found"
2026-05-17 22:08:04,604 - httpx - INFO - HTTP Request: POST http://localhost:8003/v1/completions "HTTP/1.1 404 Not Found"
2026-05-17 22:08:04,605 - httpx - INFO - HTTP Request: POST http://localhost:8004/v1/completions "HTTP/1.1 404 Not Found"
2026-05-17 22:08:04,605 - adrd_judge_runner - ERROR - Judge meditron failed: Client error '404 Not Found' for url 'http://localhost:8003/v1/completions'
For more information check: https://developer.mozilla.org/en-US/docs/Web/HTTP/Status/404
2026-05-17 22:08:04,605 - httpx - INFO - HTTP Request: POST http://localhost:8001/v1/chat/completions "HTTP/1.1 404 Not Found"
2026-05-17 22:08:04,606 - adrd_judge_runner - ERROR - Judge biomistral failed: Client error '404 Not Found' for url 'http://localhost:8002/v1/completions'
For more information check: https://developer.mozilla.org/en-US/docs/Web/HTTP/Status/404
2026-05-17 22:08:04,6

...

  [SKIPPED] Q=q_0022 — only 0 non-empty responses


2026-05-17 22:09:49,282 - httpx - INFO - HTTP Request: POST http://localhost:8001/v1/chat/completions "HTTP/1.1 404 Not Found"
2026-05-17 22:09:49,283 - adrd_judge_runner - ERROR - Judge medgemma failed: Client error '404 Not Found' for url 'http://localhost:8001/v1/chat/completions'
For more information check: https://developer.mozilla.org/en-US/docs/Web/HTTP/Status/404
2026-05-17 22:09:49,283 - adrd_judge_runner - INFO - Judge medgemma returned 0 chars in 0ms
2026-05-17 22:09:49,285 - httpx - INFO - HTTP Request: POST http://localhost:8002/v1/completions "HTTP/1.1 404 Not Found"
2026-05-17 22:09:49,286 - adrd_judge_runner - ERROR - Judge biomistral failed: Client error '404 Not Found' for url 'http://localhost:8002/v1/completions'
For more information check: https://developer.mozilla.org/en-US/docs/Web/HTTP/Status/404
2026-05-17 22:09:49,286 - adrd_judge_runner - INFO - Judge biomistral returned 0 chars in 0ms
2026-05-17 22:09:49,287 - httpx - INFO - HTTP Request: POST http://localho

  [SKIPPED] Q=q_0023 — only 0 non-empty responses


2026-05-17 22:09:49,549 - httpx - INFO - HTTP Request: POST http://localhost:8001/v1/chat/completions "HTTP/1.1 404 Not Found"
2026-05-17 22:09:49,550 - adrd_judge_runner - ERROR - Judge medgemma failed: Client error '404 Not Found' for url 'http://localhost:8001/v1/chat/completions'
For more information check: https://developer.mozilla.org/en-US/docs/Web/HTTP/Status/404
2026-05-17 22:09:49,550 - adrd_judge_runner - INFO - Judge medgemma returned 0 chars in 0ms
2026-05-17 22:09:49,551 - httpx - INFO - HTTP Request: POST http://localhost:8002/v1/completions "HTTP/1.1 404 Not Found"
2026-05-17 22:09:49,552 - adrd_judge_runner - ERROR - Judge biomistral failed: Client error '404 Not Found' for url 'http://localhost:8002/v1/completions'
For more information check: https://developer.mozilla.org/en-US/docs/Web/HTTP/Status/404
2026-05-17 22:09:49,552 - adrd_judge_runner - INFO - Judge biomistral returned 0 chars in 0ms
2026-05-17 22:09:49,562 - httpx - INFO - HTTP Request: POST http://localho

  [SKIPPED] Q=q_0024 — only 0 non-empty responses
  Mean PW: 0.00%  Std: 0.00%  n=0

ABLATION — Indistinguishable scoring variants (std < 5%):
  ⚠️  PEMAT — Patient Education Materials Assessment Tool [BINARY] std=0.0% — consider removing
  ⚠️  PEMAT — Patient Education Materials Assessment Tool [LIKERT_1_5] std=0.0% — consider removing
  ⚠️  PEMAT — Patient Education Materials Assessment Tool [SCALED_0_10] std=0.0% — consider removing
  ⚠️  HealthBench — Physician-Written Medical Response Evaluation Criteria [BINARY] std=0.0% — consider removing
  ⚠️  HealthBench — Physician-Written Medical Response Evaluation Criteria [LIKERT_1_5] std=0.0% — consider removing
  ⚠️  HealthBench — Physician-Written Medical Response Evaluation Criteria [SCALED_0_10] std=0.0% — consider removing
  ⚠️  Clinical LLM Evaluation by Expert Review [BINARY] std=0.0% — consider removing
  ⚠️  Clinical LLM Evaluation by Expert Review [LIKERT_1_5] std=0.0% — consider removing
  ⚠️  Clinical LLM Evaluation by Exper

---
## Phase 10 — Experiment 4: Box Plots

In [ ]:
# CELL 13: Experiment 4
import experiments.exp4_boxplot_agreement as exp4_mod
importlib.reload(exp4_mod)

print('Running Experiment 4: Box Plot Generation')
print('='*60)
exp4_mod.main()

pngs = sorted((ROOT / 'results' / 'figures').glob('*.png'))
print(f'\n✅ Exp4 complete: {len(pngs)} figures')
for p in pngs:
    print(f'  📊 {p.name}  ({p.stat().st_size//1024} KB)')

---
## Phase 11 — Results Summary

In [ ]:
# CELL 14: Print full results + save CSV
print('=== FINAL RESULTS — COPY TO PAPER ===\n')
print('[Exp 1] Dataset:')
for cat, n in sorted(counts.items()): print(f'  {cat}: {n}')
print(f'  TOTAL: {sum(counts.values())}')
print('\n[Exp 2] Agreement Table:')
print(df_exp2.to_markdown(index=False))
print('\n[Exp 3] Best scoring variant per rubric:')
for block in exp3_data:
    best = max(block['variants'], key=lambda v: v['mean_pairwise_agreement'])
    print(f'  {block["rubric_name"][:30]:30s} → {best["scoring_variant"]} ({best["mean_pairwise_agreement"]:.1f}%)')
print(f'\n[Exp 4] {len(pngs)} PNG figures in results/figures/')
df_exp2.to_csv(ROOT / 'results' / 'agreement_summary_table.csv', index=False)
print('\n✅ agreement_summary_table.csv saved')

---
## Phase 12 — Display Figures Inline

In [ ]:
# CELL 15: Display all PNG figures
from IPython.display import Image, display, Markdown
for png in sorted((ROOT / 'results' / 'figures').glob('*.png')):
    display(Markdown(f'### {png.stem}'))
    display(Image(filename=str(png), width=950))

---
## Phase 13 — Push Results to GitHub

In [ ]:
# CELL 16: Commit and push results
if GH_TOKEN:
    shell(f'git -C {ROOT} remote set-url origin https://{GH_TOKEN}@github.com/m22oct2000/Multi_LLM_as_Judge_Medical_AI.git')
else:
    print('⚠️  GITHUB_TOKEN not set — push will prompt for password')

COMMIT_MSG = f'HPC results (SuperPOD mkotha): all 4 experiments — {datetime.now().strftime("%Y-%m-%d %H:%M")}'

print('=== Files to commit ===')
shell(f'git -C {ROOT} status --short')

shell(f'git -C {ROOT} add results/')
shell(f'git -C {ROOT} add benchmark_dataset/')
shell(f'git -C {ROOT} commit -m "{COMMIT_MSG}"')

rc = shell(f'git -C {ROOT} push origin main')
if rc == 0:
    print('\n✅ Pushed! → https://github.com/m22oct2000/Multi_LLM_as_Judge_Medical_AI/tree/main/results')
else:
    print(f'\n⚠️  Push failed. Try: cd {ROOT} && git push origin main')

---
## Phase 14 — Cleanup

In [ ]:
# CELL 17: Kill vLLM servers and free all GPUs
import subprocess, time

print('=== GPU before cleanup ===')
subprocess.run('nvidia-smi --query-gpu=index,memory.used,memory.free --format=csv,noheader', shell=True)

if 'procs' in dir():
    for name, proc in procs.items():
        try:
            proc.terminate()
            print(f'  Terminated [{name}] PID={proc.pid}')
        except Exception:
            pass

subprocess.run('pkill -9 -f "vllm.entrypoints" 2>/dev/null || true', shell=True)
time.sleep(3)
subprocess.run(
    'kill -9 $(fuser /dev/nvidia0 /dev/nvidia1 /dev/nvidia2 /dev/nvidia3 2>/dev/null) 2>/dev/null || true',
    shell=True
)
time.sleep(4)
print('\n=== GPU after cleanup ===')
subprocess.run('nvidia-smi --query-gpu=index,memory.used,memory.free --format=csv,noheader', shell=True)
print('\n✅ Done. All 4 GPUs free. Exit srun with: exit')


---
## ✅ All Experiments Complete

| File | Use |
|---|---|
| `results/agreement_summary_table.csv` | Paper Table 2 |
| `results/figures/*.png` | Upload to Overleaf |
| `results/exp2_agreement_results.json` | Full rationales |
| `benchmark_dataset/adrd_questions.json` | Dataset reference |

**Acknowledgement for paper:**
> *"Computational resources were provided by SMU's O'Donnell Data Science and Research Computing Institute (SuperPOD cluster, 2x NVIDIA A100 80GB GPUs, allocation: xnluo_ai_chatbot_cognitive_0002)."*
